# Colab VLA Server Spike v0

Runs a temporary FastAPI VLA inference server in this Colab runtime and exposes it over an HTTPS tunnel (ngrok or cloudflared), so a **local** `RealVLAPolicyClient` (in the `physical-ai-recycling-cell` repo) can call it.

**This notebook never touches a robot.** It only serves `/health` and `/predict`. The local machine still owns the camera, PyBullet, `SafetySupervisor`, `RobotBackend`, and episode recording -- see `docs/colab_vla_server_spike.md` for the full role split.

**This is a spike/test environment, not a production server.** The session ends when Colab disconnects (idle timeout, 12/24h cap, or you closing the tab), and the tunnel URL is invalidated with it -- you'll need to re-run this notebook and update the local config again next time.

Run the cells in order: **Runtime/GPU check -> install deps -> define server -> start server + tunnel -> test curl commands**. Try `mock-action` mode first; only try `openvla-dryrun` once that works.

## 1. Runtime / GPU check

Reports what's actually available in this Colab runtime -- torch install, CUDA, GPU name/VRAM, Python version -- and prints a `recommended_mode` so you don't guess. A CPU-only or no-GPU runtime is completely fine for `health-only`/`mock-action` (the two modes this spike's success criteria actually depend on); only `openvla-dryrun` needs a GPU, and even then it degrades gracefully if one isn't available.

In [ ]:
import platform
import subprocess

print(f"python_version: {platform.python_version()}")

try:
    import torch
    torch_installed = True
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()
except ImportError:
    torch_installed = False
    torch_version = None
    cuda_available = False

print(f"torch_installed: {torch_installed}")
print(f"torch_version: {torch_version}")
print(f"cuda_available: {cuda_available}")

gpu_name = None
vram_total_gb = None
vram_free_gb = None
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    vram_total_gb = round(total_bytes / (1024 ** 3), 2)
    vram_free_gb = round(free_bytes / (1024 ** 3), 2)

print(f"gpu_name: {gpu_name}")
print(f"vram_total_gb: {vram_total_gb}")
print(f"vram_free_gb: {vram_free_gb}")

if cuda_available and (vram_free_gb or 0) >= 16:
    recommended_mode = "openvla-dryrun (GPU + enough free VRAM detected -- still expect this to be tight for a 7B VLA)"
elif cuda_available:
    recommended_mode = "mock-action (GPU detected but free VRAM looks too low for a 7B model -- openvla-dryrun will likely report model_status=not_loaded, which is fine)"
else:
    recommended_mode = "mock-action or health-only (no CUDA GPU in this runtime -- openvla-dryrun will report model_status=not_loaded by design, not crash)"

print(f"recommended_mode: {recommended_mode}")

## 2. Install dependencies

Only installs what's missing -- torch/transformers are left alone if Colab already has them (reinstalling either can churn for several minutes and isn't needed for `health-only`/`mock-action`). OpenVLA-specific dependencies are isolated in their own optional cell (2b) so skipping it never blocks `mock-action` testing.

In [ ]:
%pip install -q fastapi uvicorn pyngrok pillow requests nest_asyncio

### 2b. Optional: OpenVLA dependencies (only needed for `openvla-dryrun`)

Skip this cell entirely for `health-only`/`mock-action` testing. Only run it if the runtime/GPU check above recommended trying `openvla-dryrun`. This does **not** download the model itself yet -- that happens lazily, inside the server, the first time it's needed (see cell 4).

In [ ]:
%pip install -q transformers accelerate timm

## 3. Clone the repo and define the FastAPI server

The actual server (`openvla_server_real/colab_vla_server.py`) lives in the `physical-ai-recycling-cell` repo, not duplicated inline here -- this cell just clones/pulls the repo and imports it, so the notebook and the local codebase never drift apart. Replace `REPO_URL` with wherever you're hosting this repo (GitHub, etc.).

In [ ]:
import os
import sys

REPO_URL = "REPLACE_WITH_YOUR_REPO_URL"  # e.g. https://github.com/<you>/physical-ai-recycling-cell.git
REPO_DIR = "/content/physical-ai-recycling-cell"

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull

sys.path.insert(0, REPO_DIR)

## 4. Choose a server mode

```
health-only     no model loaded; /predict always fails with model_not_loaded (503).
                Use this first to validate the tunnel/HTTP plumbing alone.
mock-action     no real model; reuses DummyOpenVLAPolicy to return a deterministic,
                safe 7-DoF action. This is what RealVLAPolicyClient should be tested
                against first -- the actual success bar for this spike.
openvla-dryrun  best-effort real OpenVLA load (only if torch+transformers+a CUDA GPU
                are all present). If loading fails for any reason, that is recorded
                as an environment limitation (model_status=not_loaded), never a crash.
                If it does load, /predict returns the raw model output for inspection
                but still refuses to return an executable action
                (project_action_available=false, reason=action_adapter_required) --
                OpenVLA's raw output is not auto-trusted as a delta_ee_7dof action.
```

`openvla-direct` (applying raw OpenVLA output straight to the robot) is intentionally not implemented anywhere in this repo.

In [ ]:
import os

SERVER_MODE = "mock-action"  # "health-only" | "mock-action" | "openvla-dryrun"
os.environ["COLAB_VLA_SERVER_MODE"] = SERVER_MODE

from openvla_server_real.colab_vla_server import app, SERVER_MODE as loaded_mode
print(f"server_mode: {loaded_mode}")

## 5. Start the server + public tunnel

Runs uvicorn in a background thread (so this cell returns and the notebook stays usable), then opens a public HTTPS tunnel to it. Uses ngrok if `NGROK_AUTHTOKEN` is set (Colab: `Secrets` panel or `os.environ`); otherwise falls back to cloudflared's no-account quick tunnel.

In [ ]:
import threading
import time

import nest_asyncio
import uvicorn

nest_asyncio.apply()

LOCAL_PORT = 9000

config = uvicorn.Config(app, host="127.0.0.1", port=LOCAL_PORT, log_level="info")
server = uvicorn.Server(config)

server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)
print(f"local server running on http://127.0.0.1:{LOCAL_PORT}")

In [ ]:
import os

ngrok_authtoken = os.environ.get("NGROK_AUTHTOKEN")
public_url = None

if ngrok_authtoken:
    from pyngrok import ngrok

    ngrok.set_auth_token(ngrok_authtoken)
    tunnel = ngrok.connect(LOCAL_PORT, "http")
    public_url = tunnel.public_url
    print(f"tunnel: ngrok ({public_url})")
else:
    print("NGROK_AUTHTOKEN not set -- falling back to cloudflared (no account needed).")
    print("Install once per session: !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared")
    print(f"Then run in the background: !./cloudflared tunnel --url http://127.0.0.1:{LOCAL_PORT}")
    print("cloudflared prints its own https://<random>.trycloudflare.com URL to stdout/logs -- copy that as public_url below.")

if public_url:
    health_url = f"{public_url}/health"
    predict_url = f"{public_url}/predict"
    print(f"health_url: {health_url}")
    print(f"predict_url: {predict_url}")

## 6. Test curl commands

Run these from a local terminal (not this notebook) to sanity-check the tunnel before pointing `RealVLAPolicyClient` at it.

In [ ]:
if public_url:
    print("# From your local machine:\n")
    print(f"curl {public_url}/health\n")
    print(
        f"curl -X POST {public_url}/predict \\\n"
        "  -H \"Content-Type: application/json\" \\\n"
        "  -d '{\"instruction\": \"test\", \"target_object_position\": [0.4, -0.1, 0.05], \"bin_position\": [0.3, 0.35, 0.05]}'"
    )
else:
    print("Set public_url first (see cell 5's cloudflared fallback instructions).")

## 7. Update the local config and probe from your local machine

```bash
python scripts/update_colab_vla_config.py \
  --base-url <public_url printed above> \
  --config configs/real_vla_backend_colab_config.json

python -m benchmark.probe_colab_vla_server \
  --real-vla-config configs/real_vla_backend_colab_config.json \
  --with-image
```

See `docs/colab_vla_server_spike.md` for the full local probe / full-demo walkthrough.

## 8. When you're done: Colab session/shutdown notes

- **The tunnel URL dies with the Colab session.** Idle timeout, the 12/24h session cap, or just closing the tab all invalidate it immediately -- your local `RealVLAPolicyClient` will start failing its health check (and fall back to `local-dummy`/`fastapi-dummy`, if configured, which is the expected/safe behavior).
- **Re-running this notebook issues a brand-new URL.** ngrok free tier and cloudflared quick tunnels do not preserve the previous URL across restarts.
- **You must re-run `scripts/update_colab_vla_config.py` with the new URL** every time you restart this notebook -- there is no automatic sync between Colab and your local config file.
- This was never meant to run unattended for a long session -- see `docs/colab_vla_server_spike.md` for why (and what a real deployment would use instead: a persistent server, not a notebook + tunnel).